# RQ1 Assumption Checks: Self-Retrieval Accuracy

This notebook checks the basic assumptions behind the statistics used in
`experiment/run_rq1_experiment.py`.

We treat **each query song** as one observation. For each query we record:

- the rank of the query song in its own recommendation list,
- hit@1, hit@3, hit@5 (0/1 indicators),
- 1/rank (for Mean Reciprocal Rank, MRR).

The experiment script already computes HR@1, HR@3, HR@5, and MRR with
**bootstrap 95% confidence intervals** over the set of sampled queries.

Here we:

- load the actual per-query results from `experiment_results/RQ1_results.json`,
- visualise the distribution of ranks and 1/rank,
- and briefly discuss what these checks tell us about the assumptions
  behind our RQ1 metrics.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

# This notebook lives in experiment_results/assumption_checks
EXP_DIR = Path("..")
RQ1_PATH = EXP_DIR / "RQ1_results.json"

with RQ1_PATH.open(encoding="utf-8") as f:
    rq1 = json.load(f)

per_query = rq1["per_query"]

ranks = [q["rank"] for q in per_query]
hit1 = [q["hit@1"] for q in per_query]
hit3 = [q["hit@3"] for q in per_query]
hit5 = [q["hit@5"] for q in per_query]
inv_rank = [q["1/rank"] for q in per_query]

len(per_query), min(ranks), max(ranks)


In [ ]:
# Histogram of ranks
fig, ax = plt.subplots(figsize=(6, 4))
max_rank = max(ranks)
ax.hist(ranks, bins=range(1, max_rank + 2), edgecolor="black", alpha=0.8)
ax.set_xlabel("Rank of query song")
ax.set_ylabel("Number of queries")
ax.set_title("RQ1: Distribution of query-song ranks")
ax.set_xticks(range(1, max_rank + 1))
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


The histogram above shows how often the query song appears at rank 1, 2, 3,
and so on, across all sampled queries. For HR@k and MRR we are averaging
functions of these ranks over queries.

Things to look for:

- If almost all mass is at rank 1, HR@1 and MRR will be very high and
  very stable (narrow bootstrap intervals).
- If ranks are spread out, HR@1 and MRR reflect that spread; the bootstrap
  CIs will be wider.
- Any impossible values (e.g. ranks larger than the candidate set size) or
  obviously broken patterns would indicate a bug in the experiment code.


In [ ]:
# Histogram of 1/rank (per-query contribution to MRR)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(inv_rank, bins=10, edgecolor="black", alpha=0.8)
ax.set_xlabel("1 / rank")
ax.set_ylabel("Number of queries")
ax.set_title("RQ1: Distribution of per-query 1/rank values")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


The distribution of 1/rank shows how much each query contributes to MRR.

- Values near 1.0 correspond to queries where the song is ranked first.
- Lower values correspond to queries where the song appears further down
  the list.

As long as these values are within [0, 1] and the distribution looks
reasonable (no impossible values), averaging them and bootstrapping over
queries is appropriate for estimating MRR and its uncertainty.
